# Clusterización de artículos aplicables a forecast

Una vez identificados los artículos que aplican a forecast, el siguiente paso consiste en agruparlos según su comportamiento histórico de venta. El objetivo de esta etapa es clasificar los artículos en grupos homogéneos para posteriormente comparar modelos de predicción dentro de cada grupo y seleccionar el algoritmo con mejor desempeño.

La clusterización se basará en variables que describen el comportamiento de la demanda, tales como volumen vendido, frecuencia de venta, meses con venta, meses sin venta, variabilidad, intermitencia, recencia, tendencia y posible estacionalidad. Este enfoque permite diferenciar artículos de alta rotación, demanda estable, demanda intermitente, baja rotación o comportamiento variable.

Las variables iniciales propuestas para clusterizar son:

variables_cluster_final = [
    "total_unidades_vendidas_24m",(ok)
    "venta_promedio_mensual_24m",(ok)
    "conteo_facturas_24m",(ok)
    "cantidad_clientes_24m",(ok)
    "meses_con_venta_24m",(ok)
    "frecuencia_venta_24m_pct",(ok)
    "venta_promedio_reciente_last_6m",(ok)
    "ratio_venta_reciente_6m",(ok)
    "desviacion_std_venta_24m",(calcular)
    "coeficiente_variacion_venta_24m",(calcular)
    "venta_maxima_mensual_24m",(calcular)
    "venta_mediana_mensual_24m",(calcular)
    "porcentaje_meses_sin_venta_24m",(calcular)
    "max_meses_consecutivos_sin_venta",(calcular)
    "meses_desde_ultima_venta",(calcular)
    "pendiente_tendencia_24m"
]

Con estas variables se buscará capturar las dimensiones principales del comportamiento histórico de cada artículo: volumen, frecuencia, intermitencia, estabilidad, tendencia y recencia.

Posteriormente, se evaluarán modelos de clusterización como K-Means, Gaussian Mixture Models, Agglomerative Clustering y DBSCAN. La calidad de los clusters se analizará mediante métricas como Silhouette Score, Davies-Bouldin Index, Calinski-Harabasz Index y análisis visual mediante PCA.

Una vez definidos los grupos, se compararán distintos modelos de forecast por cluster, tales como Naive Forecast, Moving Average, ETS, AutoARIMA, Croston, SBA, TSB, Prophet y modelos de Machine Learning como XGBoost o LightGBM. La selección del modelo ganador se realizará mediante métricas como WAPE, MAE, RMSE, Bias y riesgo estimado de quiebre o exceso de inventario.


In [3]:
from datetime import datetime

fecha_hoy = datetime.today().strftime("%Y-%m-%d")

nombre_archivo = f"dataframe_aplica_forecast_true_rev_{fecha_hoy}.csv"

print(nombre_archivo)

dataframe_aplica_forecast_true_rev_2026-07-03.csv


In [9]:
import numpy as np
import pandas as pd
#from sklearn.linear_model import LinearRegression

ruta_aplica_forecast=f"./Datos/dataframe_aplica_forecast_true_rev_{fecha_hoy}.csv"

df=pd.read_csv(ruta_aplica_forecast,
    sep=";"
)

In [10]:
import numpy as np
import pandas as pd
#from sklearn.linear_model import LinearRegression

# ============================================================
# 1. Identificar columnas mensuales de venta
# ============================================================
# Se excluyen columnas como venta_total_2024, venta_total_2025, etc.
# porque solo queremos columnas mensuales.
# ============================================================

columnas_venta_mensual = [
    col for col in df
    if col.startswith("venta_")
    and "total" not in col.lower()
    and "promedio" not in col.lower()
    and "estimada" not in col.lower()
    and "proyecto" not in col.lower()
    and "mensual" not in col.lower()
]

# Validar columnas encontradas
print("Columnas mensuales detectadas:")
print(columnas_venta_mensual)
print(f"Total columnas mensuales: {len(columnas_venta_mensual)}")

Columnas mensuales detectadas:
['venta_enero_2024', 'venta_febrero_2024', 'venta_marzo_2024', 'venta_abril_2024', 'venta_mayo_2024', 'venta_junio_2024', 'venta_julio_2024', 'venta_agosto_2024', 'venta_septiembre_2024', 'venta_octubre_2024', 'venta_noviembre_2024', 'venta_diciembre_2024', 'venta_enero_2025', 'venta_febrero_2025', 'venta_marzo_2025', 'venta_abril_2025', 'venta_mayo_2025', 'venta_junio_2025', 'venta_julio_2025', 'venta_agosto_2025', 'venta_septiembre_2025', 'venta_octubre_2025', 'venta_noviembre_2025', 'venta_diciembre_2025', 'venta_enero_2026', 'venta_febrero_2026', 'venta_marzo_2026', 'venta_abril_2026', 'venta_mayo_2026', 'venta_junio_2026']
Total columnas mensuales: 30


In [11]:
# ============================================================
# 2. Asegurar que las columnas mensuales sean numéricas
# ============================================================

df[columnas_venta_mensual] = (
    df[columnas_venta_mensual]
    .apply(pd.to_numeric, errors="coerce")
    .fillna(0)
)

# ============================================================
# 3. Tomar últimos 24 meses disponibles
# ============================================================
# Esto permite que el cálculo sea dinámico aunque en el futuro
# existan nuevas columnas como venta_julio_2026, venta_agosto_2026, etc.
# ============================================================

columnas_venta_24m = columnas_venta_mensual[-24:]

print("Columnas usadas para análisis 24m:")
print(columnas_venta_24m)

Columnas usadas para análisis 24m:
['venta_julio_2024', 'venta_agosto_2024', 'venta_septiembre_2024', 'venta_octubre_2024', 'venta_noviembre_2024', 'venta_diciembre_2024', 'venta_enero_2025', 'venta_febrero_2025', 'venta_marzo_2025', 'venta_abril_2025', 'venta_mayo_2025', 'venta_junio_2025', 'venta_julio_2025', 'venta_agosto_2025', 'venta_septiembre_2025', 'venta_octubre_2025', 'venta_noviembre_2025', 'venta_diciembre_2025', 'venta_enero_2026', 'venta_febrero_2026', 'venta_marzo_2026', 'venta_abril_2026', 'venta_mayo_2026', 'venta_junio_2026']


In [14]:
ventas_24m = df[columnas_venta_24m]
print(type(ventas_24m))

<class 'pandas.DataFrame'>


In [15]:
# ============================================================
# 4. Calcular variables matemáticas de comportamiento histórico
# ============================================================

ventas_24m = df[columnas_venta_24m]

# ------------------------------------------------------------
# Desviación estándar de venta mensual
# ------------------------------------------------------------

df["desviacion_std_venta_24m"] = ventas_24m.std(axis=1)

# ------------------------------------------------------------
# Coeficiente de variación
# std / promedio
# Si el promedio es 0, se asigna 0 para evitar división infinita.
# ------------------------------------------------------------

# ------------------------------------------------------------
# Venta máxima mensual
# ------------------------------------------------------------

df["venta_maxima_mensual_24m"] = ventas_24m.max(axis=1)

# ------------------------------------------------------------
# Venta mediana mensual
# ------------------------------------------------------------

df["venta_mediana_mensual_24m"] = ventas_24m.median(axis=1)

# ------------------------------------------------------------
# Porcentaje de meses sin venta
# ------------------------------------------------------------

df["porcentaje_meses_sin_venta_24m"] = (
    (ventas_24m.eq(0).sum(axis=1) / len(columnas_venta_24m)) * 100
)

C:\Users\CHOPPE\AppData\Local\Temp\ipykernel_28540\1082203104.py:11: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["desviacion_std_venta_24m"] = ventas_24m.std(axis=1)
C:\Users\CHOPPE\AppData\Local\Temp\ipykernel_28540\1082203104.py:23: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["venta_maxima_mensual_24m"] = ventas_24m.max(axis=1)
C:\Users\CHOPPE\AppData\Local\Temp\ipykernel_28540\1082203104.py:29: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which

In [ ]:
# ============================================================
# 5. Funciones auxiliares
# ============================================================

def calcular_max_meses_consecutivos_sin_venta(fila):
    """
    Calcula la mayor racha consecutiva de meses con venta igual a cero.
    """
    max_racha = 0
    racha_actual = 0

    for valor in fila:
        if valor == 0:
            racha_actual += 1
            max_racha = max(max_racha, racha_actual)
        else:
            racha_actual = 0

    return max_racha


def calcular_meses_desde_ultima_venta(fila):
    """
    Calcula cuántos meses han pasado desde la última venta.
    Si nunca hubo venta en los 24 meses, devuelve 24.
    """
    valores = list(fila)

    for i, valor in enumerate(reversed(valores)):
        if valor > 0:
            return i

    return len(valores)


def calcular_pendiente_tendencia(fila):
    """
    Calcula la pendiente de tendencia usando regresión lineal simple.
    Una pendiente positiva indica tendencia creciente.
    Una pendiente negativa indica tendencia decreciente.
    """
    y = np.array(fila, dtype=float)

    if np.sum(y) == 0:
        return 0

    x = np.arange(len(y)).reshape(-1, 1)

    modelo = LinearRegression()
    modelo.fit(x, y)

    return modelo.coef_[0]


def indice_concentracion_picos_24m(fila):
    """
    Calcula un índice simple de estacionalidad.
    
    Fórmula:
    venta promedio de los 3 meses más altos / venta promedio mensual
    
    Si el resultado es alto, puede indicar concentración de ventas
    en pocos meses.
    Un valor cercano a 1 indica ventas relativamente estables.
    Un valor entre 1.5 y 3 indica presencia moderada de picos.
    Un valor mayor a 3 indica alta concentración de demanda en pocos meses.
    """
    valores = np.array(fila, dtype=float)
    promedio = valores.mean()

    if promedio == 0:
        return 0

    top_3_promedio = np.sort(valores)[-3:].mean()

    return top_3_promedio / promedio


def calcular_fuerza_estacional_24m(fila):
    """
    Calcula una medida simple de fuerza estacional basada en
    la variabilidad promedio por mes calendario.

    Compara el promedio de cada mes calendario contra el promedio general.
    correlacion_estacional_12m >= 0.70  → posible estacionalidad fuerte
     0.40 a 0.70                         → posible estacionalidad moderada
     0.20 a 0.40                         → señal débil
     < 0.20                              → sin patrón estacional claro
    """

    ventas = np.array(fila, dtype=float)

    if len(ventas) < 24:
        return np.nan

    if ventas.mean() == 0:
        return 0

    # Primeros 12 meses y segundos 12 meses
    anio_1 = ventas[:12]
    anio_2 = ventas[12:24]

    # Promedio por mes calendario entre ambos años
    promedio_mes_calendario = (anio_1 + anio_2) / 2

    # Variabilidad de los promedios mensuales contra el promedio general
    fuerza_estacional = np.std(promedio_mes_calendario) / ventas.mean()

    return fuerza_estacional


def calcular_correlacion_estacional_12m(fila):
    """
    Calcula una señal de estacionalidad comparando el patrón mensual
    de un año contra el patrón mensual del siguiente año.

    Una correlación alta indica que los meses de mayor y menor venta
    tienden a repetirse entre años.
    """

    ventas = np.array(fila, dtype=float)

    # Validar que existan al menos 24 meses
    if len(ventas) < 24:
        return np.nan

    # Primeros 12 meses vs siguientes 12 meses
    anio_1 = ventas[:12]
    anio_2 = ventas[12:24]

    # Si ambos años no tienen ventas, no hay estacionalidad evaluable
    if anio_1.sum() == 0 and anio_2.sum() == 0:
        return 0

    # Si alguno de los años no tiene variación, la correlación no es válida
    if np.std(anio_1) == 0 or np.std(anio_2) == 0:
        return 0

    correlacion = np.corrcoef(anio_1, anio_2)[0, 1]

    return correlacion